In [19]:
import os
import sys

import geopandas as gpd
import pandas as pd
import yaml

# on se place à la racine du projet
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

# paramètres communs du projet
with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

from src.variables import build_communes, build_gares, build_voyageurs_communes

# les tables propres de chaque base
communes.to_parquet("data/clean/base_finale/base_finale_communes.parquet")
gares.to_parquet("data/clean/base_finale/base_finale_gares.parquet")
gares_annees.to_parquet("data/clean/base_finale/base_finale_gares_annees.parquet")
# table SNCF de Mathilde : une ligne par gare et par année
sncf = pd.read_parquet("data/clean/sncf-merged.parquet")

In [20]:
from src.variables import build_distance_gare, build_gares_annees, build_jointure

gares = build_gares(sncf, contours)
voyageurs = build_voyageurs_communes(sncf, gares, config["annee_debut"], config["annee_fin"])
distance = build_distance_gare(contours, gares, config["crs_calcul"])
gares_annees = build_gares_annees(sncf, gares)

# la grande table : une ligne par commune
communes = build_communes(contours, population, navettes, parc, voyageurs)
communes = build_jointure(communes, distance, "distance à la gare")
print(communes.shape)
print(gares_annees.shape)

methode
code SNCF                 2716
arrondissement              42
point dans le polygone      25
Name: count, dtype: int64
gares sans commune : 0
population : 34746 communes, 0 sans correspondance (0.00 %)
navettes : 34746 communes, 0 sans correspondance (0.00 %)
parc automobile : 34746 communes, 4 sans correspondance (0.01 %)
gares et voyageurs : 34746 communes, 32296 sans correspondance (92.95 %)
distance à la gare : 34746 communes, 0 sans correspondance (0.00 %)
(34746, 36)
(19481, 6)


In [21]:
communes.to_parquet("data/clean/base_finale/base_finale_communes.parquet")
gares.to_parquet("data/clean/base_finale/base_finale_gares.parquet")
gares_annees.to_parquet("data/clean/base_finale/base_finale_gares_annees.parquet")

In [22]:
from src.variables import build_distance_gare, build_jointure

gares = build_gares(sncf, contours)
voyageurs = build_voyageurs_communes(sncf, gares, config["annee_debut"], config["annee_fin"])
distance = build_distance_gare(contours, gares, config["crs_calcul"])

# la grande table : une ligne par commune
communes = build_communes(contours, population, navettes, parc, voyageurs)
communes = build_jointure(communes, distance, "distance à la gare")
print(communes.shape)
print("communes sans distance (Corse) :", communes["dist_gare_km"].isna().sum())

methode
code SNCF                 2716
arrondissement              42
point dans le polygone      25
Name: count, dtype: int64
gares sans commune : 0
population : 34746 communes, 0 sans correspondance (0.00 %)
navettes : 34746 communes, 0 sans correspondance (0.00 %)
parc automobile : 34746 communes, 4 sans correspondance (0.01 %)
gares et voyageurs : 34746 communes, 32296 sans correspondance (92.95 %)
distance à la gare : 34746 communes, 0 sans correspondance (0.00 %)
(34746, 36)
communes sans distance (Corse) : 360


On garder les gares uniquement en France métropolitaine et en unité choisie. 

In [23]:
from src.variables import build_gares_annees

gares_annees = build_gares_annees(sncf, gares)
print(gares_annees.shape)
print(gares_annees.groupby("annee")["voyageurs"].sum())

(19481, 6)
annee
2017    2723841659
2018    2675366476
2019    2724376670
2020    1423972270
2021    2068635456
2022    2590865445
2023    2779139516
Name: voyageurs, dtype: int64
